# Lab 1: Stim Basics and Quantum Circuit Simulation

**[EQE5006] Quantum Error Correction, Spring 2026**

In this lab, you will learn how to use **stim**, a high-performance stabilizer circuit simulator developed by Google, widely used in quantum error correction (QEC) research. Starting from a single qubit, you will progressively build more complex circuits (adding entanglement and noise) while verifying the theoretical predictions from Chapters 1 and 2.

## Section 0: Getting Started with stim

**stim** is a fast simulator for stabilizer circuits (circuits composed of Clifford gates such as H, S, X, Y, Z, and CNOT). It is designed for simulating quantum error correction experiments with millions of qubits and measurement shots.

In this lab, we will use two core features:
- `stim.Circuit` — to construct quantum circuits
- `.compile_sampler().sample(shots=...)` — to sample measurement outcomes

In [ ]:
# Install stim if not already installed
!pip install stim -q

In [ ]:
import stim
import numpy as np
import matplotlib.pyplot as plt

print(f"stim version: {stim.__version__}")

### Building your first circuit

A `stim.Circuit` object represents a quantum circuit. You add operations using the `.append()` method, specifying:
1. The gate name (e.g., `"X"`, `"H"`, `"CNOT"`)
2. The target qubit(s)
3. (Optional) a parameter, such as a noise probability

Let's start with the simplest possible circuit: apply an $X$ gate to qubit 0, then measure it.

In [ ]:
# Create an empty circuit
circuit = stim.Circuit()

# Apply X gate to qubit 0
circuit.append("X", [0])

# Measure qubit 0
circuit.append("M", [0])

# Print the circuit
print(circuit)

In [ ]:
# Visualize the circuit as a text
circuit.diagram("timeline-text")

In [ ]:
# Visualize the circuit as SVG
circuit.diagram("timeline-svg")

In [ ]:
# Sample measurement outcomes
# All qubits start in the |0> state.
# X|0> = |1>, so measuring should always give outcome 1.
sampler = circuit.compile_sampler()
samples = sampler.sample(shots=1000)

print(f"Shape of samples array: {samples.shape}")  # (shots, num_measurements)
print(f"Data type: {samples.dtype}")
print(f"First 10 outcomes: {samples[:10, 0]}")
print(f"Fraction of outcome 1: {np.mean(samples[:, 0]):.4f}")  # Should be 1.0

The `sample()` method returns a NumPy array of shape `(shots, num_measurements)` with dtype `bool`. Each entry is `False` (outcome $|0\rangle$) or `True` (outcome $|1\rangle$). 

Since $X|0\rangle = |1\rangle$, every measurement returns 1.

### Statistical convergence with shot count

When a measurement outcome is probabilistic, the estimated probability converges to the true value as we increase the number of shots. Let's observe this convergence.

In [ ]:
for shots in [10, 100, 1000, 10000]:
    samples = sampler.sample(shots=shots)
    p1 = np.mean(samples[:, 0])
    print(f"shots={shots:>5d}  |  P(1) = {p1:.4f}  (theory: 1.0000)")

For a deterministic circuit like this, the result is exact regardless of shot count. The convergence behavior becomes more interesting with superposition states — we will see this in Section 1.

---

## Section 1: Single-Qubit Gate Exploration

Now that we know how to build and sample circuits, let's verify the behavior of single-qubit gates from Chapter 1. The key workflow is:

1. **Predict** the measurement outcome probabilities by hand (using theory)
2. **Simulate** the circuit with stim
3. **Compare** the empirical result with the theoretical prediction

### Hadamard gate and superposition

The Hadamard gate maps $|0\rangle$ to $|+\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$, so measuring should give outcomes 0 and 1 with equal probability.

In [ ]:
circuit = stim.Circuit()
circuit.append("H", [0])
circuit.append("M", [0])
samples = circuit.compile_sampler().sample(shots=10000)

p1 = np.mean(samples[:, 0])
print(f"P(1) = {p1:.4f}  (theory: 0.5000)")

### Gate combination: $HZH = X$

From Chapter 1, we know that $HZH = X$. Therefore, applying $H \to Z \to H$ to $|0\rangle$ should give $X|0\rangle = |1\rangle$, and the measurement outcome should always be 1.

In [ ]:
circuit = stim.Circuit()
circuit.append("H", [0])
circuit.append("Z", [0])
circuit.append("H", [0])
circuit.append("M", [0])
samples = circuit.compile_sampler().sample(shots=10000)

p1 = np.mean(samples[:, 0])
print(f"P(1) = {p1:.4f}  (theory: 1.0000)")

### Exercise 1.1

For each of the following circuits, **first predict** $P(0)$ and $P(1)$ by hand, **then verify** with stim.

**(a)** $S \to H \to M$

**(b)** $H \to S \to H \to M$

**(c)** $X \to H \to M$

In [ ]:
# Exercise 1.1(a): S -> H -> M
# Your prediction: P(0) = ???, P(1) = ???

# Your code here

In [ ]:
# Exercise 1.1(b): H -> S -> H -> M
# Your prediction: P(0) = ???, P(1) = ???

# Your code here

In [ ]:
# Exercise 1.1(c): X -> H -> M
# Your prediction: P(0) = ???, P(1) = ???

# Your code here

---

## Section 2: Two-Qubit Circuits — Bell States and Entanglement

Let's extend our circuits to two qubits. The central object is the **Bell state**, introduced in Chapter 1:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

This state is created by applying $H$ to the first qubit, then $\text{CNOT}$ with the first qubit as control and the second as target.

### Generating and measuring a Bell state

In [ ]:
bell_circuit = stim.Circuit()
bell_circuit.append("H", [0])
bell_circuit.append("CNOT", [0, 1])
bell_circuit.append("M", [0, 1])

bell_circuit.diagram("timeline-svg")

In [ ]:
samples = bell_circuit.compile_sampler().sample(shots=10000)

In [ ]:
samples

In [ ]:
# Count the occurrence of each two-qubit outcome
# Each row of `samples` is [qubit_0_outcome, qubit_1_outcome]
for q0, q1 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    count = np.sum((samples[:, 0] == q0) & (samples[:, 1] == q1))
    print(f"|{q0}{q1}>: {count:>5d} / 10000 = {count/10000:.4f}")

As expected, only outcomes $|00\rangle$ and $|11\rangle$ appear, each with probability $\approx 0.5$. The two qubits are perfectly correlated.

### Exercise 2.1

The four **Bell states** form a basis for the two-qubit Hilbert space:

- $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$
- $|\Phi^-\rangle = (|00\rangle - |11\rangle)/\sqrt{2}$
- $|\Psi^+\rangle = (|01\rangle + |10\rangle)/\sqrt{2}$
- $|\Psi^-\rangle = (|01\rangle - |10\rangle)/\sqrt{2}$

Construct circuits for the remaining three Bell states by adding $X$ or $Z$ gates to the $|\Phi^+\rangle$ circuit. Verify each one by sampling.

**Note:** $Z$-basis measurement can distinguish the $\Phi$ pair (outcomes 00/11) from the $\Psi$ pair (outcomes 01/10), but it **cannot** distinguish states within each pair (e.g., $|\Phi^+\rangle$ vs. $|\Phi^-\rangle$, since the relative phase is not observable in the $Z$-basis). Verify this and briefly explain why.

In [ ]:
# Exercise 2.1: Construct and verify the remaining 3 Bell states

# |Phi^-> = (|00> - |11>) / sqrt(2)
# Your code here

In [ ]:
# |Psi^+> = (|01> + |10>) / sqrt(2)
# Your code here

In [ ]:
# |Psi^-> = (|01> - |10>) / sqrt(2)
# Your code here

### Exercise 2.2

Construct a circuit that prepares the 3-qubit **GHZ state**:

$$|\text{GHZ}\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}$$

Verify that only outcomes $|000\rangle$ and $|111\rangle$ appear in the measurement results.

In [ ]:
# Exercise 2.2: 3-qubit GHZ state
# Your code here

---

## Section 3: Introducing Noise

Real quantum hardware is noisy. In Chapter 2, we studied quantum channels that model noise: bit-flip, phase-flip, and depolarizing channels. stim lets us insert noise operations directly into circuits.

Key noise instructions in stim:
- `X_ERROR(p)` — applies $X$ with probability $p$ (bit-flip channel)
- `Z_ERROR(p)` — applies $Z$ with probability $p$ (phase-flip channel)
- `DEPOLARIZE1(p)` — applies each of $X$, $Y$, $Z$ with probability $p/3$ (single-qubit depolarizing channel)
- `DEPOLARIZE2(p)` — two-qubit depolarizing channel

### Noise on a Bell state

Let's see what happens when we add bit-flip noise to the Bell state circuit from Section 2.

In [ ]:
# Bell state circuit WITH bit-flip noise on qubit 0
noisy_bell = stim.Circuit()
noisy_bell.append("H", [0])
noisy_bell.append("CNOT", [0, 1])
noisy_bell.append("X_ERROR", [0], 0.1)
noisy_bell.append("M", [0, 1])

noisy_bell.diagram("timeline-svg")

In [ ]:
samples = noisy_bell.compile_sampler().sample(shots=10000)

print("With X_ERROR(0.1) on qubit 0:")
for q0, q1 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    count = np.sum((samples[:, 0] == q0) & (samples[:, 1] == q1))
    print(f"  |{q0}{q1}>: {count:>5d} / 10000 = {count/10000:.4f}")

disagreement = np.mean(samples[:, 0] != samples[:, 1])
print(f"\nDisagreement rate: {disagreement:.4f}  (noise flips qubit 0 with p=0.1)")

With noise, outcomes $|01\rangle$ and $|10\rangle$ now appear: the noise breaks the perfect correlation of the Bell state. The disagreement rate should be close to $p = 0.1$.

### Exercise 3.1 — Bit-flip channel

The bit-flip channel applies $X$ with probability $p$:

$$\mathcal{N}_p(\rho) = (1 - p)\,\rho + p\,X\rho X$$

Apply `X_ERROR(p)` to a single qubit (initially in $|0\rangle$) before measurement. For each value of $p$ in $\{0.01, 0.05, 0.1, 0.2, 0.5\}$, measure the empirical error rate (fraction of outcome $1$) and compare it with the theoretical value $p$.

Present your results as a plot. You may use the template below.

In [ ]:
# Exercise 3.1: Bit-flip channel error rate

p_values = [0.01, 0.05, 0.1, 0.2, 0.5]
measured_rates = []
shots = 100000

for p in p_values:
    # TODO: Create a circuit that applies X_ERROR(p) to qubit 0, then measures.
    # TODO: Sample and compute the fraction of outcome 1.
    rate = 0.0  # Replace with your computation
    measured_rates.append(rate)

# --- Plotting template (no need to modify below) ---
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(p_values, p_values, "k--", label=r"Theory ($p$)")
ax.plot(p_values, measured_rates, "ro", markersize=8, label="Measured")
ax.set_xlabel(r"Noise parameter $p$")
ax.set_ylabel("Error rate")
ax.set_title(r"Bit-flip channel: X_ERROR($p$)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 3.2 — Depolarizing channel

The single-qubit depolarizing channel (as defined in Chapter 2) is:

$$\mathcal{N}(\rho) = (1-p)\,\rho + \frac{p}{3}\left(X\rho X + Y\rho Y + Z\rho Z\right)$$

In stim, `DEPOLARIZE1(p)` implements exactly this: it applies $X$, $Y$, or $Z$ each with probability $p/3$.

Apply `DEPOLARIZE1(p)` to a qubit initially in $|0\rangle$ before measurement. Measure the bit-flip probability (fraction of outcome 1) for the same $p$ values as Exercise 3.1.

**Question:** The bit-flip probability should be $2p/3$, not $p$. Why?

In [ ]:
# Exercise 3.2: Depolarizing channel error rate

p_values = [0.01, 0.05, 0.1, 0.2, 0.5]
measured_rates = []
shots = 100000

for p in p_values:
    # TODO: Create a circuit that applies DEPOLARIZE1(p) to qubit 0, then measures.
    # TODO: Sample and compute the fraction of outcome 1.
    rate = 0.0  # Replace with your computation
    measured_rates.append(rate)

# --- Plotting template (no need to modify below) ---
theory = [2 * p / 3 for p in p_values]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(p_values, theory, "k--", label="Theory ($2p/3$)")
ax.plot(p_values, measured_rates, "ro", markersize=8, label="Measured")
ax.set_xlabel("Noise parameter $p$")
ax.set_ylabel("Bit-flip probability")
ax.set_title("Depolarizing channel: DEPOLARIZE1($p$)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 3.3 (Bonus) — Noise on a Bell state

Insert `DEPOLARIZE1(0.1)` on the **first qubit** of the Bell state circuit (after the CNOT, before measurement). Measure the **disagreement rate**: the fraction of shots where the two qubits give different outcomes.

Compare with the ideal Bell state (disagreement rate = 0) and with the bit-flip noise case from the demo above.

In [ ]:
# Exercise 3.3 (Bonus): Depolarizing noise on Bell state
# Your code here

### Exercise 3.4 (Challenge) — Detecting errors without measuring data

Consider two data qubits (qubits 0 and 1) prepared in a Bell state $|\Phi^+\rangle$, plus one ancilla qubit (qubit 2).

Design a circuit that:
1. Prepares the Bell state on qubits 0 and 1
2. Uses CNOT gates to transfer the **parity** ($Z_1 Z_2$) of the two data qubits onto the ancilla (qubit 2)
3. Measures **only the ancilla** (not the data qubits)

Then, insert `X_ERROR(p)` on one of the data qubits (just before the CNOT gates) and observe how the ancilla measurement outcome changes.

**Questions:**
- What does the ancilla outcome tell you about whether an error occurred?
- Does this measurement disturb the data qubits? Why is this important?

In [ ]:
# Exercise 3.4 (Challenge): Parity check with ancilla
# Your code here
